# 1-Year Lifetime Value (LTV) Prediction for Digital Neobanks

This notebook demonstrates a production-style **1-Year LTV** prediction pipeline for a digital neobank.

**Design choices:**
- Scoring window philosophy: KYC + **30 days**
- Target: **1-Year** Lifetime Value
- Model: **LightGBM** with **Tweedie** objective (variance power 1.5)
- Dataset: public [FinTech Customer Life Time Value (LTV) Dataset](https://www.kaggle.com/datasets/harunrai/fintech-customer-life-time-value-ltv-dataset)

> The public dataset has very strong relationships between spend and LTV. Real-world production metrics are typically lower (Spearman ~0.65–0.75, MdAPE 7–15%). The architecture and feature approach remain the same.

## 1. Setup

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from scipy.stats import spearmanr, ks_2samp
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
%matplotlib inline

DATA_PATH = "../data/digital_wallet_ltv_dataset.csv"
print("Setup complete")

## 2. Load Data

In [ ]:
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}.\n"
        "Download it from: https://www.kaggle.com/datasets/harunrai/fintech-customer-life-time-value-ltv-dataset"
    )

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
print("LTV distribution:")
print(df["LTV"].describe())
print("\nMissing values:")
print(df.isnull().sum().sum())

## 3. Feature Engineering

In [ ]:
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["spend_to_txn_ratio"] = df["Total_Spent"] / (df["Total_Transactions"] + 1)
    df["avg_spend_per_active_day"] = df["Total_Spent"] / (df["Active_Days"] + 1)
    df["recency_score"] = 1 / (df["Last_Transaction_Days_Ago"] + 1)
    df["support_intensity"] = df["Support_Tickets_Raised"] / (df["Active_Days"] + 1)
    df["engagement_score"] = (
        df["Active_Days"] * 0.3
        + df["Loyalty_Points_Earned"] * 0.01
        + df["Referral_Count"] * 10
        + df["Customer_Satisfaction_Score"] * 5
    )
    df["high_value_txn_flag"] = (
        df["Max_Transaction_Value"] > df["Max_Transaction_Value"].quantile(0.9)
    ).astype(int)
    df["income_num"] = df["Income_Level"].map({"Low": 1, "Middle": 2, "High": 3})
    df["app_usage_num"] = df["App_Usage_Frequency"].map({"Monthly": 1, "Weekly": 2, "Daily": 3})

    cat_cols = ["Location", "Income_Level", "App_Usage_Frequency", "Preferred_Payment_Method"]
    for col in cat_cols:
        df[col] = df[col].astype("category")
    return df

df = feature_engineering(df)
print("Feature engineering done")
df.head()

## 4. Train / Validation / Test Split

In [ ]:
feature_cols = [c for c in df.columns if c not in ["Customer_ID", "LTV"]]
X = df[feature_cols]
y = df["LTV"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

## 5. Train LightGBM (Tweedie)

In [ ]:
cat_features = ["Location", "Income_Level", "App_Usage_Frequency", "Preferred_Payment_Method"]

params = {
    "objective": "tweedie",
    "tweedie_variance_power": 1.5,
    "metric": "mae",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 0.1,
    "verbose": -1,
    "random_state": 42,
    "n_jobs": -1,
}

train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_features)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data, categorical_feature=cat_features)

model = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[train_data, val_data],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100),
    ],
)

print(f"Best iteration: {model.best_iteration}")

## 6. Evaluation Metrics

In [ ]:
def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    spearman, _ = spearmanr(y_true, y_pred)
    ape = np.abs(y_true - y_pred) / (np.abs(y_true) + 1e-8)
    mdape = np.median(ape) * 100
    n = len(y_true)
    top_k = int(0.2 * n)
    true_top = set(np.argsort(y_true)[-top_k:])
    pred_top = set(np.argsort(y_pred)[-top_k:])
    precision = len(true_top & pred_top) / top_k
    return {
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "R2": round(r2, 4),
        "Spearman": round(spearman, 4),
        "MdAPE_%": round(mdape, 2),
        "Precision@top20%": round(precision, 4),
    }

y_pred_val = model.predict(X_val)
y_pred_test = model.predict(X_test)

metrics_val = compute_metrics(y_val, y_pred_val)
metrics_test = compute_metrics(y_test, y_pred_test)

print("Validation:")
for k, v in metrics_val.items():
    print(f"  {k:20s}: {v}")
print("\nTest:")
for k, v in metrics_test.items():
    print(f"  {k:20s}: {v}")

## 7. Feature Importance

In [ ]:
importance = pd.DataFrame({
    "feature": model.feature_name(),
    "gain": model.feature_importance(importance_type="gain"),
    "split": model.feature_importance(importance_type="split"),
}).sort_values("gain", ascending=False)

print(importance.head(15).to_string(index=False))

plt.figure(figsize=(9, 7))
sns.barplot(data=importance.head(15), y="feature", x="gain", orient="h", color="steelblue")
plt.title("Top 15 Feature Importance (Gain)")
plt.tight_layout()
plt.show()

## 8. Predicted vs Actual & Residuals

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred_test, alpha=0.5)
min_val = min(y_test.min(), y_pred_test.min())
max_val = max(y_test.max(), y_pred_test.max())
plt.plot([min_val, max_val], [min_val, max_val], "r--", lw=1)
plt.xlabel("Actual LTV")
plt.ylabel("Predicted LTV")
plt.title("Predicted vs Actual LTV (Test Set)")
plt.tight_layout()
plt.show()

residuals = y_test - y_pred_test
plt.figure(figsize=(8, 5))
sns.histplot(residuals, bins=50, kde=True, color="steelblue")
plt.title("Residual Distribution (Test Set)")
plt.xlabel("Actual - Predicted")
plt.tight_layout()
plt.show()

## 9. Additional Plots

Distribution of LTV, calibration-style plot and error by predicted decile.

In [ ]:
# LTV distribution
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.histplot(y_test, bins=50, kde=True, color="steelblue")
plt.title("Actual LTV Distribution (Test)")
plt.subplot(1, 2, 2)
sns.histplot(y_pred_test, bins=50, kde=True, color="coral")
plt.title("Predicted LTV Distribution (Test)")
plt.tight_layout()
plt.show()

# Mean actual vs mean predicted by decile
deciles = pd.qcut(y_pred_test, 10, labels=False, duplicates="drop")
calib = pd.DataFrame({"pred": y_pred_test, "actual": y_test, "decile": deciles})
calib_summary = calib.groupby("decile").agg({"pred": "mean", "actual": "mean"}).reset_index()

plt.figure(figsize=(7, 6))
plt.plot(calib_summary["pred"], calib_summary["actual"], "o-", label="Decile means")
plt.plot([calib_summary["pred"].min(), calib_summary["pred"].max()],
         [calib_summary["pred"].min(), calib_summary["pred"].max()], "r--", label="Perfect")
plt.xlabel("Mean Predicted LTV")
plt.ylabel("Mean Actual LTV")
plt.title("Calibration by Predicted Decile")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Statistical Tests

Kolmogorov-Smirnov test between residual distributions and basic residual diagnostics.

In [ ]:
# KS test: residuals vs normal (rough diagnostic)
residuals = y_test - y_pred_test
ks_stat, ks_p = ks_2samp(residuals, np.random.normal(residuals.mean(), residuals.std(), len(residuals)))
print(f"KS statistic (residuals vs normal): {ks_stat:.4f}")
print(f"KS p-value: {ks_p:.4f}")
print("(Low p-value suggests residuals are not perfectly normal — expected for skewed financial targets)")

# Residual vs predicted
plt.figure(figsize=(8, 5))
sns.scatterplot(x=y_pred_test, y=residuals, alpha=0.4)
plt.axhline(0, color="red", ls="--")
plt.xlabel("Predicted LTV")
plt.ylabel("Residual")
plt.title("Residuals vs Predicted")
plt.tight_layout()
plt.show()

## 11. Segment Analysis

Performance by **Location** and **Income_Level** (key demographic segments).

In [ ]:
# Prepare test set with segments
test_df = X_test.copy()
test_df["Actual"] = y_test.values
test_df["Predicted"] = y_pred_test
test_df["AbsError"] = np.abs(test_df["Actual"] - test_df["Predicted"])
test_df["APE"] = test_df["AbsError"] / (test_df["Actual"] + 1e-8)

def segment_metrics(df, segment_col):
    results = []
    for seg, g in df.groupby(segment_col, observed=True):
        if len(g) < 10:
            continue
        m = compute_metrics(g["Actual"].values, g["Predicted"].values)
        m["Segment"] = seg
        m["N"] = len(g)
        results.append(m)
    return pd.DataFrame(results)

print("=== By Location ===")
loc_metrics = segment_metrics(test_df, "Location")
display(loc_metrics.round(4))

print("\n=== By Income_Level ===")
inc_metrics = segment_metrics(test_df, "Income_Level")
display(inc_metrics.round(4))

In [ ]:
# Plot MAE by segment
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.barplot(data=loc_metrics, x="Segment", y="MAE", ax=axes[0], color="steelblue")
axes[0].set_title("MAE by Location")
axes[0].tick_params(axis="x", rotation=45)

sns.barplot(data=inc_metrics, x="Segment", y="MAE", ax=axes[1], color="coral")
axes[1].set_title("MAE by Income Level")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

# Spearman by segment
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=loc_metrics, x="Segment", y="Spearman", ax=axes[0], color="steelblue")
axes[0].set_title("Spearman by Location")
axes[0].set_ylim(0.95, 1.0)
axes[0].tick_params(axis="x", rotation=45)

sns.barplot(data=inc_metrics, x="Segment", y="Spearman", ax=axes[1], color="coral")
axes[1].set_title("Spearman by Income Level")
axes[1].set_ylim(0.95, 1.0)
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 12. Summary

- The model achieves extremely high ranking quality on this public dataset.
- Segment analysis shows consistent performance across Location and Income_Level.
- In production, expect more variation across markets and higher absolute error.
- Recommended monitoring: PSI on key features, score distribution by segment, periodic recalibration.

See `docs/TECHNICAL_DOCUMENTATION.md` for the full technical reference.